# WildChat message annotation (batched + checkpointed)

Annotates message-level rows from WildChat with the delusion-detection annotator
and appends the results to an **append-only Arrow store** in `../data/wildchat-annotations/`.

- **Batched**: messages go to the annotator in batches, fanning out concurrent requests.
- **Checkpointed**: every completed batch is written as one immutable Arrow shard;
  interrupted runs are resumed automatically (already-done messages are skipped).
- **Recompilable**: each record carries `conversation_id`, `message_idx` and a
  `content_hash`, so shards can be recombined / re-joined to WildChat later regardless
  of row order.

Config lives in the first cell; secrets come from the project `.env`.


In [1]:
import os, json, time, hashlib, shutil
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, Features, Value, concatenate_datasets, load_from_disk
from llm_delusions_annotations.annotator import Annotator, AnnotatableMessage
from llm_delusions_annotations.configs import load_annotation_configs

#TODO: Elaborate on project root step, this code is gross do I need it / can I simplify it
#TODO: What does Path.cwd do
# resolve the project root no matter where the kernel was started from
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
load_dotenv(PROJECT_ROOT / ".env")


#TODO: Read these from a config file
# --------------------------- configuration ---------------------------------
MODEL = "openai/gpt-5.4-mini-2026-03-17"
THEMES = [
    "user-metaphysical-themes",
    "user-endorses-delusion",
    "user-romantic-interest",
    "user-platonic-affinity",
    "bot-metaphysical-themes",
    "bot-endorses-delusion",
    "bot-romantic-interest",
    "bot-platonic-affinity",
]
BATCH_SIZE = 2      # messages per API round (each message spawns len(THEMES) requests)
MAX_BATCHES = 5   # None = run until done; an int = annotate this many batches per run
MAX_WORKERS = 3     # concurrent requests inside one batch

STORE_DIR = PROJECT_ROOT / "data" / "wildchat-annotations"
SHARD_DIR = STORE_DIR / "shards"

KEEP_CONTENT = True    # store the message text itself (content_hash is always stored)
KEEP_RATIONALE = False # store annotator rationales as JSON (verbose)
# ---------------------------------------------------------------------------

# Arrow schema of one annotation record - must match the dicts built in annotate_batch
STORE_FEATURES = Features({
    "conversation_id": Value("string"),
    "message_idx": Value("int64"),
    "role": Value("string"),
    "content_hash": Value("string"),
    "content": Value("string"),
    "model": Value("string"),
    "rationale": Value("string"),
    "error": Value("string"),
    "timestamp": Value("string"),
    **{theme: Value("float32") for theme in THEMES},  # None = that request failed
})

#TODO: What does assert mean / do
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY missing - put it in .env"


/home/hutchii/projects/mfmoe/MFMoe/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#TODO: What is this patch doing and why do I need it?
from huggingface_hub import login
from llm_delusions_annotations.llm_utils import client
import litellm

login(token=os.environ.get("HF_TOKEN") or None)

# strip reasoning params the chat-completions endpoint rejects (same patch as fl-bert.ipynb)
def _patched_reasoning_defaults(model, params, *, max_completion_tokens=None):
    litellm.drop_params = True
    params.setdefault("temperature", 1)
    params.setdefault("reasoning_effort", "none")

client.apply_reasoning_defaults = _patched_reasoning_defaults
print("HF + litellm patched, ready")


HF + litellm patched, ready


In [3]:
# WildChat -> per-message rows (first run downloads ~GBs; afterwards it's cached)
raw = load_dataset("allenai/WildChat")
raw_train = raw["train"].select_columns(["conversation_id", "conversation", "turn", "language"])
filtered = raw_train.filter(lambda x: x["turn"] >= 3 and x["language"] == "English")

def explode_conversations(batch):
    out = {"conversation_id": [], "message_idx": [], "role": [], "content": []}
    for cid, messages in zip(batch["conversation_id"], batch["conversation"]):
        for i, msg in enumerate(messages):
            out["conversation_id"].append(cid)
            out["message_idx"].append(i)          # position within the conversation = stable identity
            out["role"].append(msg["role"])
            out["content"].append(msg["content"])
    return out

exploded = filtered.map(explode_conversations, batched=True, remove_columns=filtered.column_names)
exploded


Dataset({
    features: ['conversation_id', 'message_idx', 'role', 'content'],
    num_rows: 708024
})

In [4]:
# --- AnnotationStore: append-only Arrow shard store (kept notebook-local for now) ---

#TODO: What does this do and why do I need it
SEP = "\x1f"
"""Field separator for checkpoint keys, chosen to never appear in real content."""


def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def make_key(conversation_id, message_idx, content) -> str:
    """Stable identity of one message, independent of row order or upstream reshuffling."""
    return SEP.join((str(conversation_id), str(message_idx), content_hash(content)))


# TODO: Need to review this class
class AnnotationStore:
    """Append-only store of annotation records as Arrow shards.

    append() writes one new immutable shard directory (saved to a .tmp dir first,
    then atomically renamed) so an interrupted run can never corrupt earlier data.
    Records are deduplicated by make_key: against the store and within the batch;
    the return value is the number of records actually written.
    """

    def __init__(self, shard_dir, features: Features):
        self.shard_dir = Path(shard_dir)
        self.features = features
        self._dataset = None
        self._keys: set[str] = set()  # keys of all annotated messages, built from shards on load
        self.shard_dir.mkdir(parents=True, exist_ok=True)
        # drop leftover partial writes from a crashed run
        for d in self.shard_dir.iterdir():
            if d.is_dir() and d.name.endswith(".tmp"):
                shutil.rmtree(d)
        for d in sorted(self.shard_dir.iterdir()):
            if d.is_dir() and (d / "state.json").exists():
                self._remember(load_from_disk(str(d)))

    def _remember(self, ds: Dataset) -> None:
        cols = ["conversation_id", "message_idx", "content"]
        for row in ds.select_columns(cols):
            self._keys.add(make_key(row["conversation_id"], row["message_idx"], row["content"]))

    @property
    def num_shards(self) -> int:
        return sum(1 for d in self.shard_dir.iterdir() if d.is_dir() and (d / "state.json").exists())

    @property
    def keys(self) -> frozenset[str]:
        """Keys of every annotated message, for checkpoint/resume lookups."""
        # TODO: What is a frozen set?
        return frozenset(self._keys)

    def __len__(self) -> int:
        return len(self._keys)

    def append(self, records: list[dict]) -> int:
        records = list({make_key(r["conversation_id"], r["message_idx"], r["content"]): r for r in records}.values())
        records = [r for r in records if make_key(r["conversation_id"], r["message_idx"], r["content"]) not in self._keys]
        if not records:
            return 0
        columns = {c: [r[c] for r in records] for c in self.features}
        idx = self.num_shards
        tmp = self.shard_dir / f"shard-{idx:05d}.tmp"
        Dataset.from_dict(columns, features=self.features).save_to_disk(str(tmp))
        tmp.rename(tmp.with_suffix(""))
        self._dataset = None
        self._keys |= {make_key(r["conversation_id"], r["message_idx"], r["content"]) for r in records}
        return len(records)

    def dataset(self) -> Dataset:
        """All records, concatenated into one Arrow dataset (cached until next append)."""
        if self._dataset is None:
            shards = [load_from_disk(str(d)) for d in sorted(self.shard_dir.iterdir())
                      if d.is_dir() and (d / "state.json").exists()]
            if not shards:
                self._dataset = Dataset.from_dict({c: [] for c in self.features}, features=self.features)
            else:
                self._dataset = concatenate_datasets(shards)
        return self._dataset

In [5]:
store = AnnotationStore(SHARD_DIR, STORE_FEATURES)

ANNOTATOR = Annotator(timeout=60, max_workers=MAX_WORKERS)

print(f"checkpoint: {len(store)} messages annotated in {store.num_shards} shards "
      f"under {SHARD_DIR.relative_to(PROJECT_ROOT)}")


checkpoint: 0 messages annotated in 0 shards under data/wildchat-annotations/shards


In [6]:
# which message roles each annotation applies to (the package silently skips the rest)
_ROLE_SCOPE = {c.spec["id"]: c.allowed_roles for c in load_annotation_configs(THEMES)}


def annotate_batch(rows: list[dict]) -> list[dict]:
    """Annotate messages in one concurrent API round -> store-ready records.

    rows: dicts with conversation_id, message_idx, role, content.
    Per-request failures land in record['error'] with scores set to None,
    so a single bad request never loses a whole batch.
    """
    results = None
    last_exc = None
    for attempt in range(3):
        try:
            results = ANNOTATOR.annotate_messages(
                [AnnotatableMessage(role=r["role"], content=r["content"]) for r in rows],
                model=MODEL,
                annotation_ids=THEMES,
            )
            break
        except Exception as exc:  # network / provider hiccup -> retry
            last_exc = exc
            wait = 2 ** attempt * 5
            print(f"batch attempt {attempt + 1} failed ({exc!r}); retrying in {wait}s")
            time.sleep(wait)
    if results is None:
        raise RuntimeError("batch failed after 3 attempts") from last_exc

    now = datetime.now(timezone.utc).isoformat(timespec="seconds")
    records = []
    for r, per_msg in zip(rows, results):
        record = {
            "conversation_id": str(r["conversation_id"]),
            "message_idx": int(r["message_idx"]),
            "role": r["role"],
            "content_hash": content_hash(r["content"]),
            "content": r["content"] if KEEP_CONTENT else "",
            "model": MODEL,
            "rationale": "",
            "error": "",
            "timestamp": now,
        }
        rationales = {}
        for theme in THEMES:
            res = per_msg.get(theme)
            scope = _ROLE_SCOPE.get(theme)
            if res is None and scope is not None and r["role"] not in scope:
                record[theme] = None  # annotation does not apply to this role -> not an error
            elif res is None:
                record[theme] = None
                record["error"] = record["error"] or "missing annotation result"
            elif res.error is not None:
                record[theme] = None
                record["error"] = record["error"] or str(res.error)[:500]
            else:
                record[theme] = float(res.score)
                #TODO: remove rational support
            if KEEP_RATIONALE and res is not None and res.rationale:
                rationales[theme] = res.rationale
        record["rationale"] = json.dumps(rationales) if rationales else ""
        records.append(record)
    return records


# small smoke test so the cell above is provably wired up before the expensive loop
_probe = annotate_batch([
    {"conversation_id": "probe", "message_idx": 0, "role": "user", "content": "What color is grass?"},
])
print(_probe[0]["user-metaphysical-themes"], _probe[0]["user-endorses-delusion"], "| error:", repr(_probe[0]["error"]))


0.0 0.0 | error: ''


In [7]:
# resumable driver: skip everything already in the store, annotate the rest
def add_key(batch):
    return {"key": [make_key(cid, i, c) for cid, i, c in zip(
        batch["conversation_id"], batch["message_idx"], batch["content"])]}

keyed = exploded.map(add_key, batched=True)
pending = keyed.filter(lambda key: key not in store.keys, input_columns="key")

n_batches = (len(pending) + BATCH_SIZE - 1) // BATCH_SIZE
total = min(MAX_BATCHES, n_batches) if MAX_BATCHES else n_batches
print(f"{len(store)} done -> {len(pending)} messages remaining ({n_batches} batches)")

pbar = tqdm(total=total, desc="annotation batches")
errors = 0
for start in range(0, len(pending), BATCH_SIZE):
    if MAX_BATCHES and pbar.n >= MAX_BATCHES:
        break
    chunk = pending.select(range(start, min(start + BATCH_SIZE, len(pending))))
    # skip keys annotated since this cell started (e.g. duplicate conversations in WildChat)
    rows = [r for r in chunk.to_list() if r["key"] not in store.keys]
    if rows:
        records = annotate_batch(rows)
        written = store.append(records)
        errors += sum(bool(r["error"]) for r in records)
        pbar.set_postfix(written=written, errors=errors)
    pbar.update(1)
pbar.close()

print(f"store now holds {len(store)} annotated messages across {store.num_shards} shards "
      f"({errors} errors this session)")
if total < n_batches:
    print(f"stopped after {total} of {n_batches} pending batches - re-run this cell to continue")


Filter: 100%|██████████| 708024/708024 [00:00<00:00, 3224948.36 examples/s]


0 done -> 708024 messages remaining (354012 batches)


annotation batches: 100%|██████████| 5/5 [00:12<00:00,  2.55s/it, errors=0, written=2]

store now holds 10 annotated messages across 5 shards (0 errors this session)
stopped after 5 of 354012 pending batches - re-run this cell to continue


In [8]:
# recompile everything into one Arrow dataset: raw material for any other use case
annotated = store.dataset()
annotated


Dataset({
    features: ['conversation_id', 'message_idx', 'role', 'content_hash', 'content', 'model', 'rationale', 'error', 'timestamp', 'user-metaphysical-themes', 'user-endorses-delusion', 'user-romantic-interest', 'user-platonic-affinity', 'bot-metaphysical-themes', 'bot-endorses-delusion', 'bot-romantic-interest', 'bot-platonic-affinity'],
    num_rows: 10
})

In [9]:
# recipes for downstream use
clean = annotated.filter(lambda err: err == "", input_columns="error")

# per-role mean scores
by_role = clean.to_pandas().groupby("role")[THEMES].mean()
by_role

# reconstruct one conversation with annotations aligned to message order:
# conv_id = clean[0]["conversation_id"]
# conv = clean.filter(lambda cid: cid == conv_id, input_columns="conversation_id").sort("message_idx")

# WildChat contains duplicated conversations -> drop duplicate keys if it matters:
# deduped = clean.to_pandas().drop_duplicates(subset=["conversation_id", "message_idx", "content_hash"])

# persist one merged copy for consumers that don't want to touch the shard dir:
# clean.save_to_disk(PROJECT_ROOT / "data" / "wildchat-annotations-compiled")


Filter: 100%|██████████| 10/10 [00:00<00:00, 3648.17 examples/s]


,user-metaphysical-themes,user-endorses-delusion,user-romantic-interest,user-platonic-affinity,bot-metaphysical-themes,bot-endorses-delusion,bot-romantic-interest,bot-platonic-affinity
role,,,,,,,,
assistant,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
user,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
